This notebook is intended to:

Extract audio features from .wav files for the Speech Emotion Recognition project.

Three types of features:

    1. Mel-spectrograms  → 2D arrays for CNN input       → saved as .npy
    2. MFCCs             → 2D sequences for LSTM input    → saved as .npy
    3. Aggregate features → 1D vectors for baseline models → saved as CSV

Usage:
    from src.features import extract_all_features
    extract_all_features(df, output_dir="outputs/features")

In [2]:
!pip install librosa


   ---- -----------------------------------  1/10 [soxr]
   -------- -------------------------------  2/10 [msgpack]
   ---------------- -----------------------  4/10 [audioop-lts]
   ---------------------------- -----------  7/10 [pooch]
   ---------------------------- -----------  7/10 [pooch]
   ---------------------------- -----------  7/10 [pooch]
   ---------------------------- -----------  7/10 [pooch]
   -------------------------------- -------  8/10 [audioread]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ ---  9/10 [librosa]
   ------------------------------------ --- 


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import numpy as np
import pandas as pd
import librosa
import warnings
warnings.filterwarnings("ignore")

# Configuration

In [4]:
SAMPLE_RATE = 22050        # Standard sample rate for librosa
DURATION = 3.0             # Fixed clip length in seconds
N_SAMPLES = int(SAMPLE_RATE * DURATION)  # 66150 samples

# Mel-spectrogram settings
N_MELS = 128               # Number of mel bands
N_FFT = 2048                # FFT window size
HOP_LENGTH = 512            # Hop between frames

# MFCC settings
N_MFCC = 40                 # Number of MFCC coefficients

# Audio loading with fixed-length padding/truncation

In [5]:
def load_audio(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """
    Load an audio file and force it to a fixed length.

    - If the clip is shorter than `duration` seconds: pad with zeros at the end
    - If the clip is longer: truncate from the end

    Returns
    -------
    y : np.ndarray of shape (N_SAMPLES,) — the raw waveform
    sr : int — sample rate
    """
    y, sr = librosa.load(file_path, sr=sr, duration=duration)

    # Pad if too short
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)), mode="constant")

    # Truncate if too long (shouldn't happen with duration param, but just in case)
    y = y[:N_SAMPLES]

    return y, sr

# Feature 1: Mel-Spectrogram (for CNNs)

In [6]:
def extract_mel_spectrogram(y, sr=SAMPLE_RATE):
    """
    Convert a waveform into a log-mel-spectrogram.

    Returns
    -------
    log_mel : np.ndarray of shape (N_MELS, T) where T is the number of time frames
              For 3 seconds at hop_length=512: T ≈ 130
    """
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    # Convert to log scale (decibels) for better dynamic range
    log_mel = librosa.power_to_db(mel, ref=np.max)

    return log_mel

In [7]:
def extract_mel_with_deltas(y, sr=SAMPLE_RATE):
    """
    Extract a 3-channel spectrogram: [log-mel, delta, delta-delta]
    This can be fed into a CNN as if it were an RGB image.

    Returns
    -------
    stacked : np.ndarray of shape (3, N_MELS, T)
    """
    log_mel = extract_mel_spectrogram(y, sr)
    delta = librosa.feature.delta(log_mel, order=1)
    delta2 = librosa.feature.delta(log_mel, order=2)

    stacked = np.stack([log_mel, delta, delta2], axis=0)
    return stacked

# Feature 2: MFCCs (for LSTMs)

In [8]:
def extract_mfccs(y, sr=SAMPLE_RATE):
    """
    Extract MFCC coefficients over time.

    Returns
    -------
    mfccs : np.ndarray of shape (N_MFCC, T) — 40 coefficients × T time frames
            Transpose to (T, N_MFCC) before feeding to LSTM
    """
    mfccs = librosa.feature.mfcc(
        y=y, sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )
    return mfccs

# Feature 3: Aggregate features (for baselines)

In [9]:
def extract_aggregate_features(y, sr=SAMPLE_RATE):
    """
    Extract a flat vector of summary statistics from an audio clip.
    These are used for traditional ML baselines (SVM, Random Forest).

    Returns
    -------
    feature_dict : dict with named features (about 170 total values)
    """
    features = {}

    # ── MFCCs: mean and std of each coefficient ──
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(N_MFCC):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfccs[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfccs[i])

    # ── MFCC deltas: rate of change ──
    mfcc_delta = librosa.feature.delta(mfccs, order=1)
    for i in range(N_MFCC):
        features[f"mfcc_delta_{i+1}_mean"] = np.mean(mfcc_delta[i])

    # ── Chroma: pitch class distribution (12 values) ──
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(12):
        features[f"chroma_{i+1}_mean"] = np.mean(chroma[i])

    # ── Spectral centroid: "brightness" of the sound ──
    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH)[0]
    features["spectral_centroid_mean"] = np.mean(spec_centroid)
    features["spectral_centroid_std"] = np.std(spec_centroid)

    # ── Spectral bandwidth: spread of the spectrum ──
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=HOP_LENGTH)[0]
    features["spectral_bandwidth_mean"] = np.mean(spec_bw)
    features["spectral_bandwidth_std"] = np.std(spec_bw)

    # ── Spectral contrast: valley-to-peak energy ratio in sub-bands ──
    spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(spec_contrast.shape[0]):
        features[f"spectral_contrast_{i+1}_mean"] = np.mean(spec_contrast[i])

    # ── Spectral rolloff: frequency below which 85% of energy lies ──
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH)[0]
    features["spectral_rolloff_mean"] = np.mean(rolloff)
    features["spectral_rolloff_std"] = np.std(rolloff)

    # ── Zero-crossing rate: how often the signal changes sign (noisiness) ──
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # ── RMS energy: overall loudness ──
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)
    features["rms_max"] = np.max(rms)

    # ── Pitch (F0): fundamental frequency ──
    # Using pyin for more robust pitch estimation
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"),
            sr=sr, hop_length=HOP_LENGTH,
        )
        f0_valid = f0[~np.isnan(f0)]
        if len(f0_valid) > 0:
            features["pitch_mean"] = np.mean(f0_valid)
            features["pitch_std"] = np.std(f0_valid)
            features["pitch_min"] = np.min(f0_valid)
            features["pitch_max"] = np.max(f0_valid)
            features["pitch_range"] = np.max(f0_valid) - np.min(f0_valid)
        else:
            features["pitch_mean"] = 0
            features["pitch_std"] = 0
            features["pitch_min"] = 0
            features["pitch_max"] = 0
            features["pitch_range"] = 0
        features["voiced_fraction"] = np.mean(voiced_flag) if voiced_flag is not None else 0
    except Exception:
        features["pitch_mean"] = 0
        features["pitch_std"] = 0
        features["pitch_min"] = 0
        features["pitch_max"] = 0
        features["pitch_range"] = 0
        features["voiced_fraction"] = 0

    # ── Tempo (BPM estimate) ──
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    # librosa may return an array; take the scalar
    features["tempo"] = float(tempo) if np.isscalar(tempo) else float(tempo[0])

    return features

# Batch extraction across the full dataset

In [10]:
def extract_all_features(df, output_dir="", include_deltas=True):
    """
    Extract all three feature types for every clip in the DataFrame.

    Saves:
        - mel_spectrograms/       → one .npy per clip (shape: N_MELS × T)
        - mel_3channel/           → one .npy per clip (shape: 3 × N_MELS × T)
        - mfccs/                  → one .npy per clip (shape: N_MFCC × T)
        - aggregate_features.csv  → one row per clip with all summary features

    Parameters
    ----------
    df : pd.DataFrame — must have 'file_path' column (from data_loader.py)
    output_dir : str — root directory for saving features
    include_deltas : bool — whether to also save 3-channel spectrograms
    """
    # Create output directories
    mel_dir = os.path.join(output_dir, "mel_spectrograms")
    mel3_dir = os.path.join(output_dir, "mel_3channel")
    mfcc_dir = os.path.join(output_dir, "mfccs")
    os.makedirs(mel_dir, exist_ok=True)
    os.makedirs(mfcc_dir, exist_ok=True)
    if include_deltas:
        os.makedirs(mel3_dir, exist_ok=True)

    aggregate_rows = []
    mel_paths = []
    mel3_paths = []
    mfcc_paths = []
    skipped = []

    total = len(df)
    print(f"Extracting features for {total} clips...")
    print(f"Output directory: {output_dir}")
    print("-" * 50)

    for idx, row in df.iterrows():
        file_path = row["file_path"]

        # Create a unique filename for this clip
        clip_id = f"{row['dataset']}_{row['speaker_id']}_{idx:05d}"

        if (idx + 1) % 500 == 0 or idx == 0:
            print(f"  Processing {idx + 1}/{total}: {os.path.basename(file_path)}")

        try:
            # Load audio
            y, sr = load_audio(file_path)

            # ── Mel-spectrogram ──
            mel = extract_mel_spectrogram(y, sr)
            mel_path = os.path.join(mel_dir, f"{clip_id}.npy")
            np.save(mel_path, mel)
            mel_paths.append(mel_path)

            # ── 3-channel mel (with deltas) ──
            if include_deltas:
                mel3 = extract_mel_with_deltas(y, sr)
                mel3_path = os.path.join(mel3_dir, f"{clip_id}.npy")
                np.save(mel3_path, mel3)
                mel3_paths.append(mel3_path)
            else:
                mel3_paths.append("")

            # ── MFCCs ──
            mfccs = extract_mfccs(y, sr)
            mfcc_path = os.path.join(mfcc_dir, f"{clip_id}.npy")
            np.save(mfcc_path, mfccs)
            mfcc_paths.append(mfcc_path)

            # ── Aggregate features ──
            agg = extract_aggregate_features(y, sr)
            agg["clip_id"] = clip_id
            agg["file_path"] = file_path
            agg["emotion"] = row["emotion"]
            agg["speaker_id"] = row["speaker_id"]
            agg["gender"] = row["gender"]
            agg["dataset"] = row["dataset"]
            aggregate_rows.append(agg)

        except Exception as e:
            print(f"  ERROR processing {file_path}: {e}")
            skipped.append(file_path)
            mel_paths.append("")
            mel3_paths.append("")
            mfcc_paths.append("")

    # Save aggregate features as CSV
    agg_df = pd.DataFrame(aggregate_rows)
    agg_csv_path = os.path.join(output_dir, "aggregate_features.csv")
    agg_df.to_csv(agg_csv_path, index=False)

    # Add feature paths back to the main DataFrame
    df = df.copy()
    df["mel_path"] = mel_paths
    df["mel3_path"] = mel3_paths
    df["mfcc_path"] = mfcc_paths

    # Save the updated DataFrame
    updated_csv_path = os.path.join(output_dir, "master_with_features.csv")
    df.to_csv(updated_csv_path, index=False)

    # Summary
    print(f"\n{'=' * 50}")
    print(f"Feature extraction complete!")
    print(f"  Mel-spectrograms:    {mel_dir}/ ({len(os.listdir(mel_dir))} files)")
    if include_deltas:
        print(f"  Mel 3-channel:       {mel3_dir}/ ({len(os.listdir(mel3_dir))} files)")
    print(f"  MFCCs:               {mfcc_dir}/ ({len(os.listdir(mfcc_dir))} files)")
    print(f"  Aggregate CSV:       {agg_csv_path} ({len(agg_df)} rows × {len(agg_df.columns)} cols)")
    print(f"  Updated DataFrame:   {updated_csv_path}")
    if skipped:
        print(f"  Skipped (errors):    {len(skipped)} clips")
    print(f"{'=' * 50}")

    # Print feature shapes for verification
    if len(mel_paths) > 0 and mel_paths[0]:
        sample_mel = np.load(mel_paths[0])
        sample_mfcc = np.load(mfcc_paths[0])
        print(f"\nFeature shapes (first clip):")
        print(f"  Mel-spectrogram: {sample_mel.shape}  (n_mels × time_frames)")
        print(f"  MFCCs:           {sample_mfcc.shape}  (n_mfcc × time_frames)")
        print(f"  Aggregate:       {len(agg_df.columns) - 6} numeric features per clip")

    return df, agg_df

In [11]:
df = pd.read_csv("master_dataframe.csv")
print(f"Loaded {len(df)} clips from master DataFrame\n")

# Extract all features
df_updated, agg_df = extract_all_features(df, output_dir="")

Loaded 4528 clips from master DataFrame

Extracting features for 4528 clips...
Output directory: 
--------------------------------------------------
  Processing 1/4528: 03-01-01-01-01-01-01.wav
  Processing 500/4528: 03-01-06-01-02-02-10.wav
  Processing 1000/4528: 03-01-03-02-02-02-20.wav
  Processing 1500/4528: OAF_good_ps.wav
  Processing 2000/4528: OAF_shawl_angry.wav
  Processing 2500/4528: OAF_goose_neutral.wav
  Processing 3000/4528: YAF_shawl_disgust.wav
  Processing 3500/4528: YAF_good_neutral.wav
  Processing 4000/4528: YAF_shawl_sad.wav
  Processing 4500/4528: KL_sa02.wav

Feature extraction complete!
  Mel-spectrograms:    mel_spectrograms/ (4528 files)
  Mel 3-channel:       mel_3channel/ (4528 files)
  MFCCs:               mfccs/ (4528 files)
  Aggregate CSV:       aggregate_features.csv (4528 rows × 163 cols)
  Updated DataFrame:   master_with_features.csv

Feature shapes (first clip):
  Mel-spectrogram: (128, 130)  (n_mels × time_frames)
  MFCCs:           (40, 130)  (